In [2]:
import numpy as np
import os
from matplotlib import pyplot as plt
import time

In [7]:
DATA_PATH = os.path.join('Dataset')
#print(os.listdir(key_points))
actions = np.array(['Good', 'Hello', 'Maybe','Thank_you', 'Yes'])

In [36]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [8]:
label_map = {label:num for num, label in enumerate(actions)}

In [9]:
label_map

{'Good': 0, 'Hello': 1, 'Maybe': 2, 'Thank_you': 3, 'Yes': 4}

In [38]:
for action in actions:
    print(action)

Thank_you
Hello
Good


In [10]:
sequences, labels = [], []

for action in os.listdir(DATA_PATH):
    if not os.path.isdir(os.path.join(DATA_PATH, action)):
        continue
    print(action)
    for sequence in os.listdir(os.path.join(DATA_PATH, action)):
        if not os.path.isdir(os.path.join(DATA_PATH, action, sequence)):
            continue
        #print(f'\t{sequence}')
        window = []
        for frame_num in range(30):
            res = np.load(os.path.join(DATA_PATH, action, str(sequence), "{}.npy".format(frame_num)))
            window.append(res)
            #print(res)
        sequences.append(window)
        labels.append(label_map[action])

Thank_you
Maybe
Good
Hello
Yes


In [40]:
np.array(sequences).shape

(300, 30, 1662)

In [41]:
np.array(labels).shape

(300,)

In [42]:
x = np.array(sequences)

In [43]:
y = to_categorical(labels).astype(int)

In [44]:
x.shape

(300, 30, 1662)

In [45]:
y

array([[1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1, 0, 0],
       [1,

In [46]:
x_train, x_test, y_train, y_test = train_test_split(x, y , test_size = 0.05)

In [47]:
y_train.shape

(285, 3)

In [48]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.models import load_model as load_model

In [49]:
log_dir = os.path.join('Logs')
tb_callback = TensorBoard(log_dir=log_dir)

In [50]:
model = Sequential()
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(30,1662)))
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(LSTM(64, return_sequences=False, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(actions.shape[0], activation='softmax'))

In [52]:
actions.shape[0]

3

In [51]:
model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])

In [53]:
model.fit(x_train, y_train, epochs=1000, callbacks=[tb_callback])

Epoch 1/1000
9/9 [==============================] - 24s 2s/step - loss: 2.9045 - categorical_accuracy: 0.3228
Epoch 2/1000
9/9 [==============================] - 21s 2s/step - loss: 1.1239 - categorical_accuracy: 0.2842
Epoch 3/1000
9/9 [==============================] - 22s 2s/step - loss: 1.0792 - categorical_accuracy: 0.3333
Epoch 4/1000
9/9 [==============================] - 23s 3s/step - loss: 1.1344 - categorical_accuracy: 0.3719
Epoch 5/1000
9/9 [==============================] - 24s 3s/step - loss: 1.1002 - categorical_accuracy: 0.3368
Epoch 6/1000
9/9 [==============================] - 24s 3s/step - loss: 1.0816 - categorical_accuracy: 0.5018
Epoch 7/1000
9/9 [==============================] - 24s 3s/step - loss: 1.0609 - categorical_accuracy: 0.3684
Epoch 8/1000
9/9 [==============================] - 26s 3s/step - loss: 1.0227 - categorical_accuracy: 0.5333
Epoch 9/1000
9/9 [==============================] - 27s 3s/step - loss: 0.9557 - categorical_accuracy: 0.5684
Epoch 10/1

KeyboardInterrupt: 

In [54]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_3 (LSTM)               (None, 30, 64)            442112    
                                                                 
 lstm_4 (LSTM)               (None, 30, 128)           98816     
                                                                 
 lstm_5 (LSTM)               (None, 64)                49408     
                                                                 
 dense_3 (Dense)             (None, 64)                4160      
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 3)                 99        
                                                                 
Total params: 596,675
Trainable params: 596,675
Non-tr

In [55]:
model.predict(x_test)

1/1 [==============================] - 0s 408ms/step


array([[1.8907216e-04, 9.1426648e-02, 9.0838426e-01],
       [5.9301168e-05, 1.4378920e-02, 9.8556179e-01],
       [4.3583317e-05, 9.8352641e-01, 1.6430026e-02],
       [1.0000000e+00, 0.0000000e+00, 0.0000000e+00],
       [8.1463797e-05, 4.4391368e-02, 9.5552725e-01],
       [1.2307113e-02, 7.8581256e-01, 2.0188034e-01],
       [1.0000000e+00, 0.0000000e+00, 0.0000000e+00],
       [9.1842283e-03, 8.1277961e-01, 1.7803621e-01],
       [1.0000000e+00, 0.0000000e+00, 0.0000000e+00],
       [1.1640232e-02, 7.8805417e-01, 2.0030560e-01],
       [1.5057131e-05, 9.9313897e-01, 6.8459590e-03],
       [1.1256136e-04, 3.7934978e-02, 9.6195245e-01],
       [3.0123020e-04, 1.8451105e-01, 8.1518769e-01],
       [6.0629183e-03, 8.0700886e-01, 1.8692817e-01],
       [8.6124483e-06, 1.5629023e-04, 9.9983513e-01]], dtype=float32)

In [56]:
model.save('actions-v0.3.4-3 signs(Thank_you, hello, good).h5')

In [57]:
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score

In [58]:
yhat = model.predict(x_test)

1/1 [==============================] - 0s 163ms/step


In [59]:
ytrue = np.argmax(y_test, axis=1).tolist()
yhat = np.argmax(yhat, axis=1).tolist()

In [60]:
multilabel_confusion_matrix(ytrue, yhat)

array([[[12,  0],
        [ 0,  3]],

       [[ 9,  0],
        [ 0,  6]],

       [[ 9,  0],
        [ 0,  6]]])

In [38]:
accuracy_score(ytrue, yhat)

0.84